[![](imagens/colab-badge.png){width="16%"}](https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/cap05/cap05_aluno.ipynb)
[![](imagens/github-badge.png){width="20%"}](https://github.com/fzampirolli/pdi-vc)

# Transformadas e Compressão

Até o momento, o processamento de imagens foi abordado exclusivamente no **domínio espacial**, onde os operadores atuam diretamente sobre as intensidades dos pixels $f(x,y)$. Neste capítulo, mudamos de paradigma e introduzimos o **domínio da frequência**, no qual a imagem é analisada não pelas suas posições, mas pela sua *taxa de variação* espacial.

A **Transformada de Fourier** permite decompor qualquer imagem em uma soma de ondas senoidais e cossensoidais, separando perfeitamente a estrutura global (baixas frequências) dos detalhes finos e ruídos (altas frequências). Através do poderoso Teorema da Convolução, operações que seriam computacionalmente custosas no domínio espacial tornam-se simples multiplicações matemáticas no domínio da frequência.

Avançando na análise espectral, introduziremos as **Wavelets**, que resolvem a principal limitação de Fourier ao oferecerem localização simultânea no espaço e na frequência (análise de multirresolução). Por fim, essas transformações matemáticas formam o alicerce para a **Compressão de Imagens**, explorando as redundâncias espaciais e a fisiologia da visão humana (redundância psicovisual) para viabilizar formatos clássicos como o JPEG, embasado na Transformada Cosseno Discreta (DCT).

## Objetivos

Ao final deste capítulo, você será capaz de:

- Compreender a **Transformada de Fourier** discreta 2D (DFT) e interpretar o espectro de magnitude e fase: frequências baixas (estrutura) vs. altas (bordas, ruído);
- Relacionar o **Teorema da Convolução** aos filtros espaciais do Cap. 3: convolução espacial $\leftrightarrow$ multiplicação em frequência;
- Projetar e aplicar **filtros no domínio da frequência**: passa-baixa (Gaussiano, Butterworth), passa-alta e passa-banda para remoção de ruído periódico;
- Compreender os fundamentos de **wavelets** e multirresolução: como diferentes escalas capturam estrutura global e detalhes locais simultaneamente (base conceitual para CNNs no Cap. 8);
- Entender os princípios de **compressão de imagens**: redundância espacial e psicovisual, transformada DCT (base do JPEG), codificação entrópica e compromisso entre qualidade $\times$ taxa de compressão;
- Comparar formatos (JPEG, PNG, WebP) e escolher a extensão adequada para cada aplicação com base nas características da imagem e do uso.

In [3]:
#| quarto-raw: true

import os, importlib, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy import ndimage

BASE_URL = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph"
for f in ["morph.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{BASE_URL}/{f}", f)

import morph
importlib.reload(morph)
from morph import mm

print("✅ Ambiente pronto")

✅ Ambiente pronto


## A Transformada Discreta de Fourier 2D (DFT)

A Transformada Discreta de Fourier 2D converte a imagem de seu domínio espacial original para o domínio das frequências espaciais. Para uma imagem digital $f(x,y)$ de dimensões $M \times N$, a DFT é definida por:

$$
F(u,v) = \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} f(x,y) e^{-j 2\pi (ux/M + vy/N)}
$$ {#eq-05-dft}

para $u = 0, 1, \ldots, M-1$ e $v = 0, 1, \ldots, N-1$. As variáveis $u$ e $v$ representam as **frequências espaciais** nos eixos horizontal e vertical.

A operação inversa (IDFT) recupera a imagem espacial original somando perfeitamente todas as ondas bidimensionais componentes:

$$
f(x,y) = \frac{1}{MN} \sum_{u=0}^{M-1} \sum_{v=0}^{N-1} F(u,v) e^{j 2\pi (ux/M + vy/N)}
$$ {#eq-05-idft}

Como o resultado $F(u,v)$ é um número complexo, ele é frequentemente analisado através de seus dois componentes fundamentais:

- **Espectro de Magnitude** $|F(u,v)|$: Determina "quanto" de cada frequência está presente na imagem. Concentra-se massivamente nas baixas frequências (centro do espectro).
- **Espectro de Fase** $\phi(u,v)$: Determina "onde" essas frequências estão localizadas espacialmente (alinhamento das ondas). Essencial para a preservação das formas e bordas geométricas.

::: {.callout-note}
### Frequências Altas vs. Baixas {.unnumbered}
- **Baixas frequências ($u,v \approx 0$):** Representam variações lentas de intensidade, como iluminação geral, fundos lisos e cores uniformes da imagem.
- **Altas frequências ($u,v \gg 0$):** Representam variações rápidas e abruptas, como bordas afiadas, texturas detalhadas e, frequentemente, ruído randômico.
:::

In [4]:
#| label: fig-05-fourier-sim
#| fig-cap: "Simulador interativo das Funções de Base da Transformada de Fourier 2D. Ajuste as frequências u (horizontal) e v (vertical) para visualizar a onda espacial correspondente."
#| echo: false
#| output: true

from IPython.display import HTML

HTML("""
<div id="ft_Root">
<style>
  #ft_Root * { box-sizing: border-box; }
  #ft_Root { font-family: sans-serif; padding: 10px; max-width: 680px; margin: 0 auto; color: #374151; }
  #ft_Root canvas { display: block; max-width: 100%; height: auto; border-radius: 6px; border: 1px solid #d1d5db; background: #ffffff; }
  .ft_panel { background: #f9fafb; border: 1px solid #e5e7eb; border-radius: 6px; padding: 8px; }
  .ft_pill { font-size: 10px; font-weight: bold; padding: 3px 8px; border-radius: 4px; border: 1px solid #7dd3fc; background: #e0f2fe; color: #0369a1; }
  .ft_grid_stats { display: grid; grid-template-columns: repeat(2, 1fr); gap: 8px; margin-bottom: 10px; }
  .ft_stat_box { background: #ffffff; border: 1px solid #e5e7eb; border-radius: 6px; padding: 10px; text-align: center; }
  .ft_stat_label { font-size: 9px; color: #6b7280; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; }
  .ft_stat_value { font-size: 18px; font-weight: bold; font-family: monospace; color: #534AB7; }
</style>

<div class="ft_panel" style="display:flex; justify-content:space-between; align-items:center; margin-bottom:10px; flex-wrap:wrap; gap:6px;">
  <span style="font-size:12px; font-weight:bold; color:#4b5563; display:flex; align-items:center; gap:6px;">
    <i class="ti ti-wave-sine" aria-hidden="true" style="font-size:14px"></i> Simulador: Funções de Base de Fourier 2D
  </span>
  <span class="ft_pill">Síntese Espacial vs Frequência</span>
</div>

<div style="display:flex; gap:12px; flex-wrap:wrap; align-items: flex-start;">
  <div style="flex:1; min-width:200px; display:flex; justify-content:center;">
    <canvas id="ft_Canvas" width="256" height="256" style="width:256px; height:256px; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);"></canvas>
  </div>

  <div style="flex:1; min-width:200px; display:flex; flex-direction:column; gap:12px;">
    <div class="ft_grid_stats">
      <div class="ft_stat_box">
        <div class="ft_stat_label">Frequência u (Horiz)</div>
        <div id="ft_valU" class="ft_stat_value">2</div>
      </div>
      <div class="ft_stat_box">
        <div class="ft_stat_label">Frequência v (Vert)</div>
        <div id="ft_valV" class="ft_stat_value">1</div>
      </div>
    </div>

    <div class="ft_panel">
      <div style="font-size:11px; font-weight:bold; margin-bottom:8px;">Controles de Frequência</div>
      
      <label style="font-size:10px; color:#6b7280; display:block; margin-bottom:4px;">Ondulações Horizontais (u)</label>
      <input type="range" id="ft_sliderU" min="0" max="16" value="2" style="width:100%; margin-bottom:12px;">
      
      <label style="font-size:10px; color:#6b7280; display:block; margin-bottom:4px;">Ondulações Verticais (v)</label>
      <input type="range" id="ft_sliderV" min="0" max="16" value="1" style="width:100%;">
    </div>
    
    <div style="font-size:10px; color:#6b7280; line-height:1.5;">
      <strong>Interpretação:</strong> Qualquer imagem digital pode ser reconstruída somando-se ponderadamente ondas como esta. O centro do espectro representa u=0 e v=0 (fundo constante).
    </div>
  </div>
</div>
</div>

<script>
(function(){
  const ft_cv = document.getElementById('ft_Canvas');
  const ft_ctx = ft_cv.getContext('2d');
  const imgData = ft_ctx.createImageData(256, 256);
  
  function ft_draw() {
    const u = parseInt(document.getElementById('ft_sliderU').value);
    const v = parseInt(document.getElementById('ft_sliderV').value);
    
    document.getElementById('ft_valU').textContent = u;
    document.getElementById('ft_valV').textContent = v;
    
    for(let y=0; y<256; y++) {
      for(let x=0; x<256; x++) {
        // Equação da onda base real de Fourier (cosseno 2D)
        let val = Math.cos(2 * Math.PI * ((u * x) / 256 + (v * y) / 256));
        let color = Math.floor((val + 1) * 127.5); // Normaliza de [-1, 1] para [0, 255]
        let idx = (y * 256 + x) * 4;
        imgData.data[idx] = color;     // R
        imgData.data[idx+1] = color;   // G
        imgData.data[idx+2] = color;   // B
        imgData.data[idx+3] = 255;     // Alpha
      }
    }
    ft_ctx.putImageData(imgData, 0, 0);
  }

  document.getElementById('ft_sliderU').addEventListener('input', ft_draw);
  document.getElementById('ft_sliderV').addEventListener('input', ft_draw);
  ft_draw();
})();
</script>
""")

### Exemplo Prático em Python

No OpenCV, a DFT pode ser calculada via `cv2.dft`, mas a biblioteca `numpy.fft` possui uma sintaxe altamente expressiva e é comumente utilizada. Como a energia das baixas frequências (estruturas grandes) é exponencialmente maior que a das altas frequências (detalhes finos), aplicamos uma transformação logarítmica para visualizar o espectro adequadamente.

Note o uso imperativo do `np.fft.fftshift`. Matematicamente, a frequência nula $(0,0)$ é calculada no topo-esquerdo da matriz de saída. O *shift* move o centro do domínio de frequência $(0,0)$ para o centro geométrico da imagem exibida, tornando a interpretação do espectro analítica e visualmente correta.

In [5]:
#| label: fig-05-dft-exemplo
#| fig-cap: "Transformada Discreta de Fourier de uma imagem de teste. O Espectro de Magnitude é dominado por altas energias no centro (baixas frequências)."
#| echo: true
#| output: true

import cv2
import numpy as np
import os

# Carregando uma imagem didática clássica
url_camera = "https://upload.wikimedia.org/wikipedia/commons/2/28/Cameraman_image.png"
caminho_cam = "imagens/camera.png"

if not os.path.exists(caminho_cam):
    os.makedirs("imagens", exist_ok=True)
    img_cam = mm.read(url_camera)
    mm.write(img_cam, caminho_cam)
else:
    img_cam = mm.read(caminho_cam)

img_gray = mm.gray(img_cam)

# 1. Cálculo da Transformada de Fourier 2D
f_transform = np.fft.fft2(img_gray)
f_shift = np.fft.fftshift(f_transform) # Centraliza a frequência 0,0

# 2. Extração dos componentes (Magnitude e Fase)
magnitude = np.abs(f_shift)
fase = np.angle(f_shift)

# Escala logarítmica para o espectro de magnitude (adição de 1 evita log(0))
mag_visual = 20 * np.log(magnitude + 1)

# Normalização para exibição gráfica
mag_visual = cv2.normalize(mag_visual, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
fase_visual = cv2.normalize(fase, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

mm.show(
    [img_gray, mag_visual, fase_visual],
    titles=["Original (Espacial)", "Espectro de Magnitude (Log)", "Espectro de Fase"],
    cols=3, figsize=(15, 5)
)

HTTPError: 404 Client Error: Not Found for url: https://upload.wikimedia.org/wikipedia/commons/2/28/Cameraman_image.png

## Teorema da Convolução e Filtragem

O pilar que sustenta o uso do domínio da frequência em PDI é o **Teorema da Convolução**, o qual afirma que a convolução espacial de duas matrizes equivale puramente à multiplicação pixel a pixel de suas transformadas de Fourier:

$$
f(x,y) * h(x,y) \quad \Longleftrightarrow \quad F(u,v) \cdot H(u,v)
$$ {#eq-05-convolucao}

onde $h(x,y)$ é um filtro espacial (como um *kernel* Gaussiano) e $H(u,v)$ é sua resposta no domínio da frequência. Isso permite projetar filtros com cortes exatos nas frequências:

- **Filtros Passa-Baixa (Suavização):** Preservam as baixas frequências (centro) e atenuam as altas (bordas da matriz DFT). Efeito prático análogo ao Borramento Gaussiano do Cap. 3.
- **Filtros Passa-Alta (Realce de Bordas):** Preservam as altas frequências e removem o centro. Efeito prático análogo à filtragem Laplaciana e extração de contornos.

Abaixo, a @fig-05-filtros exemplifica o projeto de uma máscara ideal passa-baixa e a reconstrução espacial via IDFT.

In [ ]:
#| label: fig-05-filtros
#| fig-cap: "Filtragem no domínio da frequência: Máscara Passa-Baixa Ideal e reconstrução suavizada."
#| echo: true
#| output: true

linhas, colunas = img_gray.shape
c_y, c_x = linhas // 2, colunas // 2

# 1. Projetando um filtro Passa-Baixa Circular (Raio = 40)
raio_corte = 40
mask_pb = np.zeros((linhas, colunas), np.uint8)
cv2.circle(mask_pb, (c_x, c_y), raio_corte, 1, -1)

# 2. Aplicação do filtro (Multiplicação em Frequência)
f_shift_filtrado = f_shift * mask_pb

# 3. Transformada Inversa (Recuperando a imagem espacial)
f_ishift = np.fft.ifftshift(f_shift_filtrado) # Desfaz o shift
img_reconstruida = np.fft.ifft2(f_ishift)     # IDFT
img_reconstruida = np.abs(img_reconstruida)   # Extrai a magnitude real

mask_visual = (mask_pb * 255).astype(np.uint8)
img_reconstruida_vis = cv2.normalize(img_reconstruida, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

mm.show(
    [img_gray, mask_visual, img_reconstruida_vis],
    titles=["Original", "Filtro Passa-Baixa (H)", "Reconstrução (Suavizada)"],
    cols=3, figsize=(15, 5)
)

::: {.callout-important appearance="default" icon=false}
### O Fenômeno de *Ringing* (Anelamento) {.unnumbered}
Note a presença de anéis circulares ecoando pelas bordas e limites do homem na imagem reconstruída acima. Isso se deve ao corte abrupto ("degrau") do filtro passa-baixa ideal. O uso de filtros com atenuação gradual, como o **Filtro de Butterworth** ou **Gaussiano**, elimina esse artefato, justificando a dominância das gaussianas na filtragem espacial clássica.
:::

## Multirresolução e Wavelets

A Transformada de Fourier indica a magnitude exata das frequências na imagem, contudo, falha ao tentar reportar **onde** no espaço (pixels) essa frequência ocorreu. Para contornar isso, a Análise de Multirresolução e a **Transformada Wavelet Discreta (DWT)** decompõem o sinal através de pequenas funções em forma de onda ancoradas no espaço, garantindo uma análise combinada de espaço e frequência.

Na DWT 2D, a imagem passa iterativamente por bancos de filtros (passa-baixa e passa-alta horizontais e verticais), fragmentando-a em 4 sub-bandas na primeira escala:

- **LL (Aproximação):** Filtro passa-baixa duplo. Captura a essência da imagem em resolução menor.
- **LH (Detalhe Horizontal):** Bordas ao longo do eixo horizontal.
- **HL (Detalhe Vertical):** Bordas ao longo do eixo vertical.
- **HH (Detalhe Diagonal):** Bordas diagonais de altíssima frequência.

O processo pode ser repetido recursivamente sobre o bloco **LL**, criando a pirâmide de resoluções que fundamenta a compressão JPEG 2000 moderna.

In [ ]:
#| label: fig-05-dwt
#| fig-cap: "Transformada Wavelet Discreta (DWT) nível 1 utilizando a base de Haar. A imagem é segregada perfeitamente entre aproximação global e os três domínios direcionais de borda."
#| echo: true
#| output: true

import pywt

# 1. Aplicação da DWT 2D usando a wavelet de Haar
coeffs2 = pywt.dwt2(img_gray, 'haar')
LL, (LH, HL, HH) = coeffs2

# Funções de normalização para contraste visual adequado das altas frequências
def norm_wav(w_img):
    return cv2.normalize(np.abs(w_img), None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

mm.show(
    [norm_wav(LL), norm_wav(LH), norm_wav(HL), norm_wav(HH)],
    titles=["LL (Aproximação)", "LH (Bordas Horizontais)", "HL (Bordas Verticais)", "HH (Bordas Diagonais)"],
    cols=4, figsize=(18, 5)
)

## Compressão de Imagens e a DCT

Transformadas bidimensionais compõem o motor matemático dos **Padrões de Compressão de Imagens**. O processo de compressão busca mitigar três principais redundâncias:

1. **Redundância Espacial (Correlação de Pixels):** Pixels vizinhos costumam possuir a mesma cor/intensidade.
2. **Redundância Psicovisual:** O sistema visual humano (HVS) é muito mais sensível a variações gerais de brilho do que a detalhes cromáticos ínfimos de alta frequência. Alterar levemente a cor em texturas caóticas passa despercebido.
3. **Redundância de Codificação:** Distribuição otimizada de símbolos utilizando códigos curtos para eventos frequentes (ex. Codificação de Huffman).

O formato padrão **JPEG**, soberano na internet, aplica a **Transformada Cosseno Discreta (DCT)** em pequenos blocos espaciais de 8x8 pixels. Diferente de Fourier, a DCT opera unicamente com números reais e compacta extraordinariamente a energia do sinal no canto superior-esquerdo do bloco (termos DC e baixas frequências).

Na fase subsequente, chamada de **Quantização**, os coeficientes DCT de alta frequência são drasticamente divididos e arredondados para zero, acarretando perda irreversível de dados, porém invisível aos nossos olhos. É isso que consolida o JPEG como um algoritmo *lossy* (com perdas) fenomenalmente eficiente.

Abaixo, simulamos a codificação JPEG modificando o parâmetro interno de compressão (Qualidade de 1 a 100) via OpenCV.

In [ ]:
#| label: fig-05-jpeg-compress
#| fig-cap: "Avaliação da compressão JPEG com perda progressiva de qualidade. A taxa de compressão reflete o ganho em armazenamento."
#| echo: true
#| output: true

original_size = img_gray.nbytes # Tamanho em bytes da matriz bruta 8-bit

imgs_jpeg = []
titles_jpeg = []

for q in [90, 50, 10, 2]:
    # Imencode converte o Numpy Array num buffer de memória codificado como JPEG
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), q]
    success, encimg = cv2.imencode('.jpg', img_gray, encode_param)
    
    compressed_size = len(encimg)
    ratio = original_size / compressed_size
    
    # Imdecode traz o buffer comprimido de volta ao domínio espacial da matriz OpenCV
    decimg = cv2.imdecode(encimg, cv2.IMREAD_GRAYSCALE)
    
    imgs_jpeg.append(decimg)
    titles_jpeg.append(f"Q={q} | Taxa: {ratio:.1f}:1")

mm.show(
    imgs_jpeg,
    titles=titles_jpeg,
    cols=4, figsize=(18, 5)
)

## Resumo

Neste capítulo, formalizamos a transição entre as abordagens de processamento digital clássico (espacial vs. frequência) e seus desdobramentos computacionais para otimização de dados:

* **Domínio da Frequência:** Analisar imagens computando *taxas de variações* revela nuances invisíveis ao espaço, como ruídos matriciais e harmônicos estruturais puros.
* **Transformadas (Fourier e Wavelet):** Foram introduzidas tanto a magnitude analítica baseada em ondas infinitesimais contínuas (DFT), quanto sua versão adaptada para resolução bidimensional com localização espacial (Wavelets Haar/Daubechies).
* **Filtragem Acelarada (Convolução):** O arcabouço espectral viabiliza que pesadas convoluções de matrizes com núcleos imensos sejam convertidas em operações multiplicativas de altíssima rapidez.
* **Fundamentação de Compressão:** A engenharia que sustenta a internet visual contemporânea fundamenta-se nas redundâncias do olho humano e na transformada local DCT (Padrão JPEG).

O próximo capítulo abordará **Redes Neurais e Detecção Automática**, expandindo as filtragens analíticas projetadas até aqui para núcleos autônomos treináveis baseados em aprendizado de máquina profundo (*Deep Learning*).

## 🤖 Uso do NotebookLM como Tutor Complementar

Nesta edição, incentiva-se o uso do **NotebookLM** como ferramenta complementar de aprendizagem. Baseado em inteligência artificial, o sistema utiliza exclusivamente os documentos fornecidos pelo autor como fonte de conhecimento, produzindo respostas alinhadas ao conteúdo e à abordagem adotada ao longo do livro.

::: {.callout-important appearance="default" icon=false}

### 🎓 Estude com o Tutor Inteligente {.unnumbered}

[🚀 ACESSAR NOTEBOOKLM: CAPÍTULO 05](https://notebooklm.google.com/notebook/cap05)
:::

## Lista de Exercícios

1. **(15%)** Carregue duas imagens completamente distintas. Calcule a DFT de ambas, inverta o Espectro de Fase entre elas mantendo seus respectivos Espectros de Magnitude originais e proceda com a IDFT. Discuta os resultados visuais em termos de percepção geométrica vs iluminação.
2. **(15%)** Projete uma matriz máscara correspondente a um **Filtro Passa-Alta Ideal Circular** centrada (eliminando o DC da imagem de Fourier). Reconstrua a imagem e avalie se as bordas resultantes emulam uma máscara de Prewitt/Laplaciana espacial clássica.
3. **(15%)** Implemente o **Filtro Passa-Baixa Butterworth** na DFT 2D (cujo contorno de queda é gradual regido por uma potência paramétrica). Compare o fenômeno de *Ringing* (anelamento) do seu filtro com a borda bruta gerada pelo corte ideal da @fig-05-filtros.
4. **(15%)** Execute uma DWT em dois níveis de profundidade (escala 2) sobre a foto de uma face, aplique *Thresholding* rígido zerando todos os coeficientes de detalhe da malha e retorne a imagem pela `idwt2`. Avalie o impacto de compressão.
5. **(20%)** Aplique a compressão base do modelo JPEG e crie um gráfico bidimensional relacionando o fator Q de Qualidade e a Taxa Final de Compressão (CR) e outro com a razão Q e o MSE (*Mean Squared Error*) gerado para avaliar qualitativamente a distorção introduzida.
6. **(20%)** Utilizando a biblioteca OpenCV, escreva uma pequena sub-rotina que aplique de forma ingênua a função real 1D de Cosseno para construir e exibir a biblioteca base de 8x8 funções espaciais que alicerçam a transformada DCT, plotando-as numa grade de `matplotlib`.

## Referências do Capítulo {.unnumbered}

A fundamentação teórica deste capítulo baseia-se nas seguintes obras:

* @gonzalez2018digital para os preceitos fundamentais da matemática aplicada à DFT e introdução dos conceitos de base de filtragem harmônica.
* @mallat1999wavelet para o mapeamento e caracterização completa do uso e modelagem das bases multirresolutivas de Wavelets.
* @wallace1992jpeg para o descritivo normativo fundamental da infraestrutura de algoritmos DCT para o processo de compactação de sinais gráficos visuais do JPEG.
* @szeliski2022 para análises robustas das frequências atreladas a abordagens modernas de reconhecimento e interpretação na área de Visão Computacional.